In [1]:
import pandas as pd
import janitor
import os
import json
from tableone import TableOne

In [2]:
df_repos = pd.read_csv("../input/repo_baselines.csv")
repo_ids = set(df_repos["id"].dropna().astype(int))
print(f"Repos: {len(df_repos)}")
print(f"Treated: {df_repos['treated'].sum()}")
df_repos.head(3)

Repos: 582
Treated: 97


,id,node_id,name,full_name,private,owner,html_url,description,fork,url,...,treated2,_merge,year_created,license_str,n_topics,size_mb,is_org,user,owner_str,description_size
0,632697072.0,R_kgDOJbYw8A,ezfinpy,renanmoretto/ezfinpy,False,"{'login': 'renanmoretto', 'id': 103861667, 'no...",https://github.com/renanmoretto/ezfinpy,NaN,False,https://api.github.com/repos/renanmoretto/ezfinpy,...,0,left_only,2023,MIT,0,0.016602,0,renanmoretto,renanmoretto,0
1,629920730.0,R_kgDOJYvT2g,statplot,dingyizhao/statplot,False,"{'login': 'dingyizhao', 'id': 46778380, 'node_...",https://github.com/dingyizhao/statplot,Common plot code used in astrophysics,False,https://api.github.com/repos/dingyizhao/statplot,...,0,left_only,2023,MIT,0,0.002930,0,dingyizhao,dingyizhao,37
2,611058264.0,R_kgDOJGwCWA,imgutils,deepghs/imgutils,False,"{'login': 'deepghs', 'id': 126587470, 'node_id...",https://github.com/deepghs/imgutils,A convenient and user-friendly anime-style ima...,False,https://api.github.com/repos/deepghs/imgutils,...,0,left_only,2023,MIT,3,168.523438,1,deepghs,deepghs,141


In [3]:
GH_ARCHIVE_PATH = "../../gh_archive/gh_events_2023_parquet/"
months = sorted([f for f in os.listdir(GH_ARCHIVE_PATH) if f.endswith(".parquet")])
# months

RELEVANT_TYPES = {
    "ForkEvent",
    "WatchEvent",
    "IssuesEvent",
    "PushEvent",
    "PullRequestEvent",
    "ReleaseEvent",
}

dfs = []
for fname in months:
    df = pd.read_parquet(
        os.path.join(GH_ARCHIVE_PATH, fname),
        columns=["created_at", "type", "repo_id", "repo_name", "payload"],
    )
    df = df[df["repo_id"].isin(repo_ids) & df["type"].isin(RELEVANT_TYPES)]
    dfs.append(df)
    print(f"{fname}: {len(df)} events")

df_events = pd.concat(dfs, ignore_index=True)
df_events["created_at"] = pd.to_datetime(df_events["created_at"], utc=True)
print(f"\nTotal: {len(df_events)} events across {df_events['repo_id'].nunique()} repos")

gh_events_2023-01.parquet: 4057 events
gh_events_2023-02.parquet: 4089 events
gh_events_2023-03.parquet: 5363 events
gh_events_2023-04.parquet: 15995 events
gh_events_2023-05.parquet: 15805 events
gh_events_2023-06.parquet: 7990 events
gh_events_2023-07.parquet: 6612 events
gh_events_2023-08.parquet: 5389 events
gh_events_2023-09.parquet: 5364 events
gh_events_2023-10.parquet: 6020 events
gh_events_2023-11.parquet: 6086 events
gh_events_2023-12.parquet: 5058 events

Total: 87828 events across 565 repos


In [4]:
def get_issues_action(payload):
    try:
        p = payload if isinstance(payload, dict) else json.loads(payload)
        return p.get("action")
    except:
        return None


df_events = df_events.assign(
    issues_action=lambda df_: (
        df_.loc[df_["type"] == "IssuesEvent", "payload"].apply(get_issues_action)
    )
)
df_events.head()

,created_at,type,repo_id,repo_name,payload,issues_action
0,2023-01-01 18:00:08+00:00,PushEvent,584172068,avialxee/scoobi,"{""push_id"":12150772201,""size"":1,""distinct_size...",NaN
1,2023-01-09 17:26:25+00:00,PushEvent,578636915,GaNiziolek/FoccoERPy,"{""push_id"":12226591912,""size"":1,""distinct_size...",NaN
2,2023-01-01 18:02:51+00:00,PushEvent,584172068,avialxee/scoobi,"{""push_id"":12150785681,""size"":1,""distinct_size...",NaN
3,2023-01-05 09:25:32+00:00,PushEvent,262207814,huaweicloud/huaweicloud-sdk-python-v3,"{""push_id"":12186456013,""size"":1,""distinct_size...",NaN
4,2023-01-06 19:30:55+00:00,ForkEvent,18892209,tartley/colorama,"{""forkee"":{""id"":586041464,""node_id"":""R_kgDOIu5...",NaN


In [5]:
TREATMENT_START = pd.Timestamp("2023-05-12", tz="UTC")
TREATMENT_END = pd.Timestamp("2023-05-20", tz="UTC")

df_pre = df_events[df_events["created_at"] < TREATMENT_START]
df_treat = df_events[
    (df_events["created_at"] >= TREATMENT_START)
    & (df_events["created_at"] < TREATMENT_END)
]
df_post = df_events[df_events["created_at"] >= TREATMENT_END]

In [6]:
def agg_period(df, suffix):
    # All event types except IssuesEvent
    counts = (
        df[df["type"] != "IssuesEvent"]
        .groupby(["repo_id", "type"])
        .size()
        .unstack(fill_value=0)
    )

    # Issues separately
    issues = df[df["type"] == "IssuesEvent"]
    counts["issues_opened"] = (
        issues[issues["issues_action"] == "opened"].groupby("repo_id").size()
    )
    counts["issues_closed"] = (
        issues[issues["issues_action"] == "closed"].groupby("repo_id").size()
    )

    counts = counts.fillna(0).astype(int)
    counts.columns = [f"{c}_{suffix}" for c in counts.columns]
    return counts


pre_counts = agg_period(df_pre, "pre")
treat_counts = agg_period(df_treat, "treat")
post_counts = agg_period(df_post, "post")

In [7]:
df_all = (
    df_repos.select_columns(["id", "full_name", "treated", "treated2"])
    .assign(repo_id=lambda df_: df_["id"].astype("int64"))
    .set_index("repo_id")
    .join(pre_counts)
    .join(treat_counts)
    .join(post_counts)
    .fillna(0)
    .pipe(lambda df_: df_.astype({c: "int64" for c in df_.columns if c != "full_name"}))
    .clean_names()
)
df_all.head()

/home/lsys/social_proof_stars/github_exp/venv_socialproof/lib/python3.10/site-packages/pandas_flavor/register.py:161: FutureWarning: This function will be deprecated in a 1.x release. Please use `jn.select` instead.
  return method(self._obj, *args, **kwargs)


,id,full_name,treated,treated2,forkevent_pre,pullrequestevent_pre,pushevent_pre,releaseevent_pre,watchevent_pre,issues_opened_pre,...,watchevent_treat,issues_opened_treat,issues_closed_treat,forkevent_post,pullrequestevent_post,pushevent_post,releaseevent_post,watchevent_post,issues_opened_post,issues_closed_post
repo_id,,,,,,,,,,,,,,,,,,,,,
632697072,632697072,renanmoretto/ezfinpy,0,0,0,0,10,0,1,0,...,0,0,0,0,0,1,0,0,0,0
629920730,629920730,dingyizhao/statplot,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
611058264,611058264,deepghs/imgutils,0,0,0,26,232,2,1,0,...,1,0,0,3,66,481,13,50,3,0
511216783,511216783,imbus/robotframework-robosapiens,0,0,0,0,23,0,0,0,...,0,0,0,1,0,76,0,6,1,1
630769416,630769416,joshlay/amdgpu_stats,0,0,0,57,100,16,1,0,...,0,0,0,0,4,14,2,0,0,0


In [8]:
pre_vars = [
    "watchevent_pre",
    "pushevent_pre",
    "pullrequestevent_pre",
    "issues_opened_pre",
    "issues_closed_pre",
    "forkevent_pre",
    "releaseevent_pre",
]
df_all.groupby("treated2")[pre_vars].mean().round(2)

,watchevent_pre,pushevent_pre,pullrequestevent_pre,issues_opened_pre,issues_closed_pre,forkevent_pre,releaseevent_pre
treated2,,,,,,,
0,11.85,29.45,10.51,1.86,1.62,1.70,1.72
1,4.62,29.97,11.77,1.78,1.45,1.22,1.88
2,30.88,56.04,33.58,7.96,8.25,7.62,0.88


In [9]:
df_all.query("full_name!='deepmind/mujoco'").groupby("treated2")[pre_vars].mean().round(
    2
)

,watchevent_pre,pushevent_pre,pullrequestevent_pre,issues_opened_pre,issues_closed_pre,forkevent_pre,releaseevent_pre
treated2,,,,,,,
0,11.85,29.45,10.51,1.86,1.62,1.70,1.72
1,4.62,29.97,11.77,1.78,1.45,1.22,1.88
2,10.00,49.52,34.00,2.78,2.78,4.74,0.78


In [10]:
_cols = [
    "forkevent_treat",
    "watchevent_treat",
    "issues_opened_treat",
    "issues_closed_treat",
    "pushevent_treat",
    "pullrequestevent_treat",
    "releaseevent_treat",
]
df_all.groupby("treated2")[_cols].mean().round(2)

,forkevent_treat,watchevent_treat,issues_opened_treat,issues_closed_treat,pushevent_treat,pullrequestevent_treat,releaseevent_treat
treated2,,,,,,,
0,0.16,1.28,0.23,0.11,2.62,0.89,0.13
1,0.14,20.18,0.07,0.07,2.30,0.63,0.07
2,0.46,66.83,0.50,0.38,2.38,0.79,0.00


In [11]:
df_all.groupby("treated2")[_cols].mean().round(2)

,forkevent_treat,watchevent_treat,issues_opened_treat,issues_closed_treat,pushevent_treat,pullrequestevent_treat,releaseevent_treat
treated2,,,,,,,
0,0.16,1.28,0.23,0.11,2.62,0.89,0.13
1,0.14,20.18,0.07,0.07,2.30,0.63,0.07
2,0.46,66.83,0.50,0.38,2.38,0.79,0.00


In [12]:
df_all.to_stata("../output/repo_gharchive_events.dta", write_index=False, version=117)

In [13]:
t1 = TableOne(
    df_all,
    columns=pre_vars,
    groupby="treated",
    nonnormal=pre_vars,
    pval=True,
    htest_name=True,
)

t1

Grouped by treated                                                         
                                                 Missing         Overall               0                1 P-Value
n                                                                    582             485               97        
watchevent_pre, median [Q1,Q3]                         0   0.0 [0.0,2.0]   0.0 [0.0,2.0]    0.0 [0.0,3.0]   0.645
pushevent_pre, median [Q1,Q3]                          0  9.0 [4.0,25.0]  9.0 [4.0,24.0]  10.0 [4.0,29.0]   0.733
pullrequestevent_pre, median [Q1,Q3]                   0   0.0 [0.0,3.0]   0.0 [0.0,2.0]    0.0 [0.0,6.0]   0.904
issues_opened_pre, median [Q1,Q3]                      0   0.0 [0.0,0.0]   0.0 [0.0,0.0]    0.0 [0.0,1.0]   0.266
issues_closed_pre, median [Q1,Q3]                      0   0.0 [0.0,0.0]   0.0 [0.0,0.0]    0.0 [0.0,0.0]   0.963
forkevent_pre, median [Q1,Q3]                          0   0.0 [0.0,0.0]   0.0 [0.0,0.0]    0.0 [0.0,0.0]   0.743
releaseevent_pre, median [Q1,Q3]                       0   0.0 [0.0,2.0]   0.0 [0.0,2.0]    0.0 [0.0,2.0]   0.660

In [14]:
t2 = TableOne(
    df_all,
    columns=pre_vars,
    groupby="treated2",
    nonnormal=pre_vars,
    pval=True,
    htest_name=True,
)

t2

Grouped by treated2                                                                                         
                                                  Missing         Overall               0               1                2 P-Value            Test
n                                                                     582             485              73               24                        
watchevent_pre, median [Q1,Q3]                          0   0.0 [0.0,2.0]   0.0 [0.0,2.0]   0.0 [0.0,3.0]    0.0 [0.0,4.2]   0.848  Kruskal-Wallis
pushevent_pre, median [Q1,Q3]                           0  9.0 [4.0,25.0]  9.0 [4.0,24.0]  9.0 [3.0,25.0]  17.5 [4.8,82.2]   0.152  Kruskal-Wallis
pullrequestevent_pre, median [Q1,Q3]                    0   0.0 [0.0,3.0]   0.0 [0.0,2.0]   0.0 [0.0,0.0]   2.0 [0.0,29.5]   0.019  Kruskal-Wallis
issues_opened_pre, median [Q1,Q3]                       0   0.0 [0.0,0.0]   0.0 [0.0,0.0]   0.0 [0.0,0.0]    0.0 [0.0,3.2]   0.132  Kruskal-Wallis
issues_closed_pre, median [Q1,Q3]                       0   0.0 [0.0,0.0]   0.0 [0.0,0.0]   0.0 [0.0,0.0]    0.0 [0.0,2.5]   0.158  Kruskal-Wallis
forkevent_pre, median [Q1,Q3]                           0   0.0 [0.0,0.0]   0.0 [0.0,0.0]   0.0 [0.0,0.0]    0.0 [0.0,0.5]   0.618  Kruskal-Wallis
releaseevent_pre, median [Q1,Q3]                        0   0.0 [0.0,2.0]   0.0 [0.0,2.0]   0.0 [0.0,2.0]    0.0 [0.0,1.0]   0.564  Kruskal-Wallis